In [ ]:
import os
import re
import json
from datetime import datetime, timezone
from pathlib import Path
from llama_cloud import LlamaCloud

: 

In [20]:
from dotenv import load_dotenv

# Load the variables from the .env file into the system environment
load_dotenv()

True

# Parsing PDFs

In [5]:


class SEBILlamaParser:
    def __init__(self):
        """
        Initializes the production-grade LlamaParse client wrapper.
        Expects LLAMA_CLOUD_API_KEY to be set in environment variables.
        """
        api_key = os.environ.get("LLAMA_CLOUD_API_KEY")
        if not api_key:
            raise ValueError("CRITICAL: LLAMA_CLOUD_API_KEY environment variable is not set.")
        
        # Initialize the official unified LlamaCloud client wrapper
        self.client = LlamaCloud(api_key=api_key)
        print("🚀 SEBI LlamaParse Layer Initialized successfully.")

    def extract_metadata_from_md(self, markdown_text: str) -> dict:
        """
        Parses metadata from the extracted Markdown output.
        """
        # Isolate the first ~3000 characters (typically the first page)
        first_page_chunk = markdown_text[:3000]
        
        # 1. Capture standard SEBI Circular formats (e.g., SEBI/HO/IMD/CIR/P/2024/12)
        circ_match = re.search(r'SEBI/[A-Z0-9_/:-]+', first_page_chunk)
        circular_number = circ_match.group(0) if circ_match else "UNKNOWN_CIRCULAR"
        
        # 2. Extract Date formatting variants common in Indian legal updates
        date_match = re.search(r'([A-Za-z]+ \d{1,2}, \d{4}|\d{2}[./-]\d{2}[./-]\d{4})', first_page_chunk)
        issuance_date = date_match.group(0) if date_match else "UNKNOWN_DATE"
                
        # 3. Deduce legal category using rule-based classification heuristics
        category = "General Legal/SEBI Master"
        text_lower = first_page_chunk.lower()
        if "investment advisor" in text_lower or " ia " in text_lower:
            category = "Investment Advisors (IA)"
        elif "research analyst" in text_lower or " ra " in text_lower:
            category = "Research Analysts (RA)"
            
        return {
            "circular_number": circular_number,
            "issuance_date": issuance_date,
            "category": category
        }

    def parse_document(self, pdf_path: str, output_dir: str):
        """
        Ingests, orchestrates cloud OCR/parsing, and dumps structured outputs.
        """
        filename = os.path.basename(pdf_path)
        doc_id = os.path.splitext(filename)[0]
        
        if not os.path.exists(pdf_path):
            print(f"❌ Error: File not found at {pdf_path}")
            return None

        print(f"🔄 Ingesting '{filename}' into LlamaParse (Agentic Engine)...")

        try:
            # 1. Stream file payload into secure storage
            with open(pdf_path, "rb") as f:
                uploaded_file = self.client.files.create(file=f, purpose="parse")
            
            # 2. Trigger asynchronous Agentic Parse job 
            # Output configurations ensure optimized image/table routing
            parse_result = self.client.parsing.parse(
                file_id=uploaded_file.id,
                tier="agentic", # Uses premium AI visual logic for dense grids/images
                version="latest",
                expand=["markdown"]
            )
            
            # 3. Extract compiled markdown payload from completed run
            full_markdown_text = ""
            if parse_result.markdown and parse_result.markdown.pages:
                full_markdown_text = "\n\n".join([page.markdown for page in parse_result.markdown.pages if page.markdown])
            
            if not full_markdown_text.strip():
                print(f"⚠️ Warning: LlamaParse yielded empty text for {filename}.")
            
            # 4. Programmatic legal metadata identification
            extracted_meta = self.extract_metadata_from_md(full_markdown_text)
            
            final_payload = {
                "doc_id": doc_id,
                "title": doc_id.replace("_", " ").title(),
                "circular_number": extracted_meta["circular_number"],
                "issuance_date": extracted_meta["issuance_date"],
                "category": extracted_meta["category"],
                "parsed_at": datetime.utcnow().isoformat(),
                "source_engine": "LlamaParse-Agentic-V2"
            }
            
            # 5. Commit outputs directly to disk
            os.makedirs(output_dir, exist_ok=True)
            
            md_path = os.path.join(output_dir, f"{doc_id}.md")
            with open(md_path, "w", encoding="utf-8") as f:
                f.write(full_markdown_text)
                
            meta_path = os.path.join(output_dir, f"{doc_id}_meta.json")
            with open(meta_path, "w", encoding="utf-8") as f:
                json.dump(final_payload, f, indent=4)
                
            print(f"✅ Successfully Processed: {doc_id} -> Saved to {output_dir}")
            return md_path, final_payload

        except Exception as e:
            print(f"❌ Production Exception encountered while processing {filename}: {str(e)}")
            return None

# --- Local Execution Verification Sandbox ---
if __name__ == "__main__":
    # Ensure you have a directory containing your files
    PDF_INPUT_FILE = "./PDFs/RA_REG.pdf" 
    OUTPUT_DIRECTORY = "./parsed_rag_data"
    
    # Run pipeline pass
    try:
        parser_engine = SEBILlamaParser()
        parser_engine.parse_document(pdf_path=PDF_INPUT_FILE, output_dir=OUTPUT_DIRECTORY)
    except Exception as ex:
        print(f"Initialization Halt: {ex}")

🚀 SEBI LlamaParse Layer Initialized successfully.
🔄 Ingesting 'RA_REG.pdf' into LlamaParse (Agentic Engine)...
✅ Successfully Processed: RA_REG -> Saved to ./parsed_rag_data


C:\Users\ishan\AppData\Local\Temp\ipykernel_30372\3088944025.py:88: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "parsed_at": datetime.utcnow().isoformat(),


In [21]:
# Configuration Settings
INPUT_DIR = "./PDFs"                  # Folder where your raw PDFs live
OUTPUT_DIR = "./processed_markdowns"  # Where parsed RAG outputs will be saved

class JupyterSEBIFolderParser:
    def __init__(self, api_key: str, output_dir: str):
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Initialize the official unified LlamaCloud client wrapper
        self.client = LlamaCloud(api_key=api_key)
        print("🚀 SEBI LlamaParse Folder Engine Initialized successfully.")

    def extract_legal_metadata(self, markdown_text: str) -> dict:
        """
        Parses metadata from the extracted Markdown output.
        Prioritizes SEBI circular numbers; falls back to LAD regulation tracking if absent.
        """
        # Isolate the first ~4000 characters (typically the first page)
        first_page_chunk = markdown_text[:4000]
        
        # 1. Capture standard SEBI Circular formats (e.g., SEBI/HO/IMD/CIR/P/2024/12)
        circ_match = re.search(r'SEBI/[A-Z0-9_/:-]+', first_page_chunk)
        
        if circ_match:
            circular_number = circ_match.group(0)
            doc_type = "SEBI Circular"
        else:
            # 2. Fallback: Search for Legal Affairs Department Regulation codes (e.g., LAD-NRO, LAD-NGO)
            lad_match = re.search(r'LAD-[A-Z0-9_/:-]+', first_page_chunk)
            if lad_match:
                circular_number = lad_match.group(0)
                doc_type = "SEBI Regulation (LAD)"
            else:
                circular_number = "UNKNOWN_REGULATORY_ID"
                doc_type = "General Legal"
        
        # 3. Extract Date formatting variants common in Indian legal updates
        date_match = re.search(r'([A-Za-z]+ \d{1,2}, \d{4}|\d{2}[./-]\d{2}[./-]\d{4})', first_page_chunk)
        issuance_date = date_match.group(0) if date_match else "UNKNOWN_DATE"
                
        # 4. Deduce legal category using rule-based classification heuristics
        category = "General Legal/SEBI Master"
        text_lower = first_page_chunk.lower()
        if "investment advisor" in text_lower or " ia " in text_lower:
            category = "Investment Advisors (IA)"
        elif "research analyst" in text_lower or " ra " in text_lower:
            category = "Research Analysts (RA)"
            
        return {
            "circular_number": circular_number,
            "document_type": doc_type,
            "issuance_date": issuance_date,
            "category": category
        }

    async def parse_document_async(self, pdf_path: Path):
        """
        Ingests, orchestrates cloud OCR/parsing asynchronously for a single file task.
        """
        filename = pdf_path.name
        doc_id = pdf_path.stem
        
        try:
            # 1. Stream file payload into secure storage
            with open(pdf_path, "rb") as f:
                uploaded_file = self.client.files.create(file=f, purpose="parse")
            
            # 2. Trigger asynchronous Agentic Parse job 
            parse_result = self.client.parsing.parse(
                file_id=uploaded_file.id,
                tier="agentic", 
                version="latest",
                expand=["markdown"]
            )
            
            # 3. Extract compiled markdown payload from completed run
            full_markdown_text = ""
            if parse_result.markdown and parse_result.markdown.pages:
                full_markdown_text = "\n\n".join([page.markdown for page in parse_result.markdown.pages if page.markdown])
            
            if not full_markdown_text.strip():
                print(f"⚠️ Warning: LlamaParse yielded empty text for {filename}.")
                return
            
            # 4. Programmatic legal metadata identification with regulatory logic fallback
            extracted_meta = self.extract_legal_metadata(full_markdown_text)
            
            final_payload = {
                "doc_id": doc_id,
                "title": doc_id.replace("_", " ").title(),
                "circular_number": extracted_meta["circular_number"],
                "document_type": extracted_meta["document_type"],
                "issuance_date": extracted_meta["issuance_date"],
                "category": extracted_meta["category"],
                "parsed_at": datetime.now(timezone.utc).isoformat(),
                "source_engine": "LlamaParse-Agentic-V2"
            }
            
            # 5. Commit outputs directly to disk
            md_path = os.path.join(self.output_dir, f"{doc_id}.md")
            with open(md_path, "w", encoding="utf-8") as f:
                f.write(full_markdown_text)
                
            meta_path = os.path.join(self.output_dir, f"{doc_id}_meta.json")
            with open(meta_path, "w", encoding="utf-8") as f:
                json.dump(final_payload, f, indent=4)
                
            print(f"✨ Parsed: {doc_id} -> Assigned ID: {extracted_meta['circular_number']}")
            
        except Exception as e:
            print(f"❌ Exception encountered while processing {filename}: {str(e)}")

In [ ]:
import asyncio

# Retrieve Key
api_key = os.environ.get("LLAMA_CLOUD_API_KEY")
if not api_key:
    raise ValueError("CRITICAL ERROR: Please export LLAMA_CLOUD_API_KEY before running this cell.")

# Instantiate pipeline engine
folder_parser = JupyterSEBIFolderParser(api_key=api_key, output_dir=OUTPUT_DIR)

# Gather all target PDF files in the target path
pdf_targets = list(Path(INPUT_DIR).glob("*.pdf"))

if not pdf_targets:
    print(f"❌ No PDF documents found in target folder: {INPUT_DIR}")
else:
    print(f"📂 Found {len(pdf_targets)} PDFs. Dispatching parallel parsing tasks...")
    
    # asyncio.gather launches processing on all files concurrently inside Jupyter
    await asyncio.gather(*(folder_parser.parse_document_async(pdf) for pdf in pdf_targets))
    
    print(f"\n✅ Pipeline Complete! All components successfully exported to: '{OUTPUT_DIR}'")

🚀 SEBI LlamaParse Folder Engine Initialized successfully.
📂 Found 15 PDFs. Dispatching parallel parsing tasks...
✨ Parsed: IA+RA+Cir -> Assigned ID: UNKNOWN_REGULATORY_ID
✨ Parsed: IA_Cir -> Assigned ID: UNKNOWN_REGULATORY_ID
✨ Parsed: IA_cir2 -> Assigned ID: SEBI/HO/MIRSD/MIRSD-P
✨ Parsed: IA_cir3 -> Assigned ID: SEBI/HO/MIRSD/
✨ Parsed: IA_cir4 -> Assigned ID: SEBI/HO/MIRSD/MIRSD-P
✨ Parsed: IA_master -> Assigned ID: UNKNOWN_REGULATORY_ID
✨ Parsed: IA_RA_Cir2 -> Assigned ID: SEBI/HO/MIRSD/
✨ Parsed: IA_RA_cir3 -> Assigned ID: SEBI/HO/MIRSD/


: 

In [ ]:

# Configuration Settings
INPUT_DIR = "./PDFs"                  # Folder where your raw PDFs live
OUTPUT_DIR = "./processed_markdowns"  # Where parsed RAG outputs will be saved

class JupyterSEBIFolderParser:
    def __init__(self, api_key: str, output_dir: str):
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Initialize the official unified LlamaCloud client wrapper
        self.client = LlamaCloud(api_key=api_key)
        print("🚀 SEBI LlamaParse Folder Engine Initialized successfully.")

    def extract_legal_metadata(self, markdown_text: str) -> dict:
        """
        Parses metadata from the extracted Markdown output.
        Prioritizes case-sensitive SEBI circular formats, falling back to LAD regulations.
        """
        # Focus on the first ~4000 characters (typically covers the entire first page)
        first_page_chunk = markdown_text[:4000]
        
        # 1. Updated Regex to capture lowercase values like 'PoD'
        circ_match = re.search(r'SEBI/[a-zA-Z0-9_/:-]+', first_page_chunk)
        
        if circ_match:
            circular_number = circ_match.group(0)
            doc_type = "SEBI Circular"
        else:
            # 2. Fallback: Search for Legal Affairs Department Regulation codes (e.g., LAD-NRO, LAD-NGO)
            lad_match = re.search(r'LAD-[a-zA-Z0-9_/:-]+', first_page_chunk)
            if lad_match:
                circular_number = lad_match.group(0)
                doc_type = "SEBI Regulation (LAD)"
            else:
                circular_number = "UNKNOWN_REGULATORY_ID"
                doc_type = "General Legal"
        
        # 3. Extract Date formatting variants common in Indian legal updates
        date_match = re.search(r'([A-Za-z]+ \d{1,2}, \d{4}|\d{2}[./-]\d{2}[./-]\d{4})', first_page_chunk)
        issuance_date = date_match.group(0) if date_match else "UNKNOWN_DATE"
                
        # 4. Deduce legal category using rule-based classification heuristics
        category = "General Legal/SEBI Master"
        text_lower = first_page_chunk.lower()
        if "investment advisor" in text_lower or " ia " in text_lower:
            category = "Investment Advisors (IA)"
        elif "research analyst" in text_lower or " ra " in text_lower:
            category = "Research Analysts (RA)"
            
        return {
            "circular_number": circular_number,
            "document_type": doc_type,
            "issuance_date": issuance_date,
            "category": category
        }

    async def parse_document_async(self, pdf_path: Path):
        """
        Ingests and orchestrates cloud OCR/parsing asynchronously.
        If a local Markdown file exists, skips cloud processing and updates metadata locally.
        """
        filename = pdf_path.name
        doc_id = pdf_path.stem
        
        # Compute expected local destinations beforehand
        md_path = os.path.join(self.output_dir, f"{doc_id}.md")
        meta_path = os.path.join(self.output_dir, f"{doc_id}_meta.json")
        
        try:
            # --- CACHE CHECK CONDITION ---
            if os.path.exists(md_path):
                print(f"💾 Found local Markdown cache for '{filename}'. Bypassing Cloud API call.")
                with open(md_path, "r", encoding="utf-8") as f:
                    full_markdown_text = f.read()
                source_engine = "Local-Cache-Re-parse"
            else:
                # --- NO CACHE FOUND: INITIATE LLAMAPARSE TRANSACTION ---
                print(f"🔄 Ingesting '{filename}' into LlamaParse (Agentic Engine)...")
                
                # 1. Stream file payload into secure storage
                with open(pdf_path, "rb") as f:
                    uploaded_file = self.client.files.create(file=f, purpose="parse")
                
                # 2. Trigger asynchronous Agentic Parse job 
                parse_result = self.client.parsing.parse(
                    file_id=uploaded_file.id,
                    tier="agentic", 
                    version="latest",
                    expand=["markdown"]
                )
                
                # 3. Extract compiled markdown payload from completed run
                full_markdown_text = ""
                if parse_result.markdown and parse_result.markdown.pages:
                    full_markdown_text = "\n\n".join([page.markdown for page in parse_result.markdown.pages if page.markdown])
                
                if not full_markdown_text.strip():
                    print(f"⚠️ Warning: LlamaParse yielded empty text for {filename}.")
                    return
                
                # Save structural Markdown file locally for caching next time
                with open(md_path, "w", encoding="utf-8") as f:
                    f.write(full_markdown_text)
                    
                source_engine = "LlamaParse-Agentic-V2"
            
            # --- UNIFIED LOCAL PARSING & LEDGER DUMP ---
            # This executes for both cached and newly pulled files
            extracted_meta = self.extract_legal_metadata(full_markdown_text)
            
            final_payload = {
                "doc_id": doc_id,
                "title": doc_id.replace("_", " ").title(),
                "circular_number": extracted_meta["circular_number"],
                "document_type": extracted_meta["document_type"],
                "issuance_date": extracted_meta["issuance_date"],
                "category": extracted_meta["category"],
                "parsed_at": datetime.now(timezone.utc).isoformat(),
                "source_engine": source_engine
            }
            
            # Write tracking metadata json manifest files
            with open(meta_path, "w", encoding="utf-8") as f:
                json.dump(final_payload, f, indent=4)
                
            print(f"✨ Processed: {doc_id} -> ID: {extracted_meta['circular_number']} [{source_engine}]")
            
        except Exception as e:
            print(f"❌ Exception encountered while processing {filename}: {str(e)}")

: 

In [ ]:
import asyncio

# Retrieve API Key from your local system configuration environment
api_key = os.environ.get("LLAMA_CLOUD_API_KEY")
if not api_key:
    raise ValueError("CRITICAL ERROR: Please export LLAMA_CLOUD_API_KEY before running this cell.")

# Instantiate pipeline engine
folder_parser = JupyterSEBIFolderParser(api_key=api_key, output_dir=OUTPUT_DIR)

# Gather all target PDF files in the target path
pdf_targets = list(Path(INPUT_DIR).glob("*.pdf"))

if not pdf_targets:
    print(f"❌ No PDF documents found in target folder: {INPUT_DIR}")
else:
    print(f"📂 Found {len(pdf_targets)} PDFs. Dispatching parallel parsing pipeline execution...")
    
    # asyncio.gather launches processing on all files concurrently
    await asyncio.gather(*(folder_parser.parse_document_async(pdf) for pdf in pdf_targets))
    
    print(f"\n✅ Pipeline Sync Complete! Check elements inside folder: '{OUTPUT_DIR}'")